In [1]:
import pandas as pd
import numpy as np
import json, time, xmltodict, os, urllib
from pathlib import Path

In [2]:
# task specific imports
from Bio import Entrez

In [3]:
base_dir = Path("../")
parsed_data_dir = base_dir / "parsed_data"
raw_data_dir = base_dir / "raw_data"

In [4]:
# To use the api, NCBI asks we give our emails, it's optional
Entrez.email = ""    

In [5]:
DOWNLOAD_NCBI_FILES = False

## Series level metadata

Plant Cistrome data deposited to <a href="https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE60141">GSE60141</a>: <i>A Comprehensive Atlas of Arabidopsis Regulatory DNA \[DAP-seq\]</i>
 

In [6]:
SERIES_ACCESSION = "GSE60141"
series_metadata_file = raw_data_dir / f"series_{SERIES_ACCESSION}_metadata_dump.json"
sample_metadata_file = raw_data_dir / f"series_{SERIES_ACCESSION}_sample_metadata_dump.json"
gsm_metadata_file = raw_data_dir / f"series_{SERIES_ACCESSION}_gsm_metadata_dump.json"

In [7]:
if (series_metadata_file ).exists():
    with open(series_metadata_file, "r") as f: series_data = json.load(f)
    samples = series_data["esummary"][0]['Samples']
    print(SERIES_ACCESSION)
    print(len(samples))

else:
    samples = {}
    series_data = {}    
    handle = Entrez.esearch(db="gds", term=f"{SERIES_ACCESSION}[Accession] AND gse[Filter]", field="acc", retmax=1)
    series_result = Entrez.read(handle)
    series_uid = series_result["IdList"][0]
    print(SERIES_ACCESSION, series_uid)
    
    series_data["esearch"] = series_result
    
    # Get samples from the series
    handle = Entrez.esummary(db="gds", id=series_uid, retmax=1000)
    series_result = Entrez.read(handle)
    samples = series_result[0]['Samples']
    print(len(samples))
    
    series_data["esummary"] = series_result

    with open(raw_data_dir / series_metadata_file, "w") as f: json.dump(series_data, f)


GSE60141
936


In [8]:
all_sample_details = {}
all_gsm_details = {}

## Sample level metadata
### Attempt to load previously collected data

In [9]:
if (sample_metadata_file).exists():
    with open(sample_metadata_file, "r") as f: all_sample_details = json.load(f)
else:
    all_sample_details = {}
print(len(all_sample_details))

936


In [10]:
if (gsm_metadata_file).exists():
    with open(gsm_metadata_file, "r") as f: all_gsm_details = json.load(f)  
else:
    all_gsm_details = {}
print(len(all_gsm_details))    

936


### Fetch anything missing

In [11]:
failed_samples = []
for s in samples:
    if not (s["Accession"] in all_gsm_details):
        try:
            # GEO dataset
            # Get GSM UID
            handle = Entrez.esearch(db="gds", term=f"{s['Accession']}[Accession] AND gsm[Filter]", field="acc", retmax=1)
            sample = Entrez.read(handle)
            sample_uid = sample["IdList"][0]
            # Get sample details
            handle = Entrez.esummary(db="gds", id=sample_uid, retmax=100)
            sample_details = Entrez.read(handle)[0]
            all_gsm_details[s["Accession"]] = sample_details
            time.sleep(1)
        except:
            print(s)
            failed_samples.append(s)
    if not (s["Accession"] in all_sample_details):
        try:
            # BioSample
            # Get sample UID        
            handle = Entrez.esearch(db="biosample", term=f"{s['Accession']}[Accession]", field="acc", retmax=1)
            sample = Entrez.read(handle)
            sample_uid = sample["IdList"][0]
    
            # Get sample details
            handle = Entrez.esummary(db="biosample", id=sample_uid, retmax=100)
            sample_details = Entrez.read(handle)
            
            all_sample_details[s["Accession"]] = sample_details
    
        except:
            print(s)
            failed_samples.append(s)


In [12]:
with open(sample_metadata_file, "w") as f: json.dump(all_sample_details, f)
with open(gsm_metadata_file, "w") as f: json.dump(all_gsm_details, f)

In [13]:
failed_samples

[]

## Prepare metadata table

In [14]:
keep_sample_details = {}

keep_attributes = [
    "gene id", 
    "protein affinity tag", 
    "expression system", 
    "dna source", 
    "protein", 
    "protein family", 
    "replicate", 
    "subset"
]

for acc, sample_details in all_sample_details.items():
    document_summary = sample_details['DocumentSummarySet']['DocumentSummary'][0]
    sample_data = xmltodict.parse(document_summary['SampleData'])
    attributes = sample_data['BioSample']['Attributes']['Attribute']
    for d in attributes:
        if d['@attribute_name'] in keep_attributes:
            all_gsm_details[acc][d['@attribute_name']] = d['#text']    


In [15]:
len(all_gsm_details)

936

In [16]:
df = pd.DataFrame.from_dict(all_gsm_details, orient="index")
df.index.name = "accession"
df = df[['title', 'summary', 'gene id', 'taxon', 'protein', 'protein family', 'protein affinity tag', 'expression system', 'dna source', 'replicate', 'subset']]
df.replace("", pd.NA, inplace=True)
print(df.shape)
df.head(3)

(936, 11)


,title,summary,gene id,taxon,protein,protein family,protein affinity tag,expression system,dna source,replicate,subset
accession,,,,,,,,,,,
GSM1925316,G2like_tnt.At5g29000_col_a,young leaf,AT5G29000,Arabidopsis thaliana,At5g29000,G2like,Halo,tnt,col,a,motif+TFBS
GSM1925233,C2H2_tnt.At3g60580_col_a,young leaf,AT3G60580,Arabidopsis thaliana,At3g60580,C2H2,Halo,tnt,col,a,motif+TFBS
GSM1925067,AP2EREBP_tnt.CRF4_col_a,young leaf,AT4G27950,Arabidopsis thaliana,CRF4,AP2EREBP,Halo,tnt,col,a,motif


In [17]:
df[df["gene id"] == "AT3G23210"]

,title,summary,gene id,taxon,protein,protein family,protein affinity tag,expression system,dna source,replicate,subset
accession,,,,,,,,,,,
GSM1925917,bHLH_tnt.bHLH34_col_b,young leaf,AT3G23210,Arabidopsis thaliana,bHLH34,bHLH,Halo,tnt,col,b,motif
GSM1925761,bHLH_tnt.bHLH34_colamp_a,young leaf,AT3G23210,Arabidopsis thaliana,bHLH34,bHLH,Halo,tnt,colamp,a,motif
GSM1925916,bHLH_tnt.bHLH34_col_a,young leaf,AT3G23210,Arabidopsis thaliana,bHLH34,bHLH,Halo,tnt,col,a,motif


## Filter table to samples with applicable DAPseq data

In [18]:
df = df[df["taxon"] == "Arabidopsis thaliana"]
df.shape

(934, 11)

In [19]:
df[df["subset"]=="motif+TFBS"].shape

(621, 11)

In [20]:
# df = df[df["subset"]=="motif+TFBS"]

In [21]:
df["replicate"].value_counts()

replicate
a      792
b       60
v3a     37
v3b     26
v31     15
v3h      1
v3d      1
v35      1
v3e      1
Name: count, dtype: int64

## Merge metadata for replicates
Some samples (replicates) are merged, need to identify the "common" study title, and thus the common file. 

In [22]:
def get_common(x):
    # x = list(x)
    if len(x) == 1:
        return x.iloc[0]
    elif x.iloc[0] == x.iloc[1]:
        return x.iloc[0]
    else:
        print(f"{x.index.values}\t{x.iloc[0]}\t{x.iloc[1]}")
        return None

agg = {
    c: get_common for c in ['summary', 'taxon', 'protein', 'protein family', 'protein affinity tag', 'expression system', 'subset']
}

agg['replicate'] = ','.join
agg['title'] = list
agg['accession'] = list
# agg['replicate'] = list


df = df.reset_index().groupby(["gene id", "dna source"]).agg(agg)

In [23]:
df['common title'] = df['title'].apply(lambda x: os.path.commonprefix(list(x)).rstrip("v3").rstrip("_"))

In [24]:
df.shape

(872, 11)

In [25]:
df["subset"].value_counts()

subset
motif+TFBS    568
motif         304
Name: count, dtype: int64

In [26]:
df["protein affinity tag"].value_counts()

protein affinity tag
Halo    870
GST       2
Name: count, dtype: int64

In [27]:
df["expression system"].value_counts()

expression system
tnt      870
ecoli      2
Name: count, dtype: int64

In [28]:
df.head()

summary                 taxon    protein  \
gene id   dna source                                                
AT1G01060 col         young leaf  Arabidopsis thaliana       LHY1   
          colamp      young leaf  Arabidopsis thaliana       LHY1   
AT1G01250 col         young leaf  Arabidopsis thaliana  AT1G01250   
          colamp      young leaf  Arabidopsis thaliana  AT1G01250   
AT1G01720 col         young leaf  Arabidopsis thaliana      ATAF1   

                     protein family protein affinity tag expression system  \
gene id   dna source                                                         
AT1G01060 col            MYBrelated                 Halo               tnt   
          colamp         MYBrelated                 Halo               tnt   
AT1G01250 col              AP2EREBP                 Halo               tnt   
          colamp           AP2EREBP                 Halo               tnt   
AT1G01720 col                   NAC                 Halo               tnt   

                          subset replicate                              title  \
gene id   dna source                                                            
AT1G01060 col         motif+TFBS         a        [MYBrelated_tnt.LHY1_col_a]   
          colamp      motif+TFBS         a     [MYBrelated_tnt.LHY1_colamp_a]   
AT1G01250 col         motif+TFBS         a     [AP2EREBP_tnt.AT1G01250_col_a]   
          colamp           motif         a  [AP2EREBP_tnt.AT1G01250_colamp_a]   
AT1G01720 col         motif+TFBS         a              [NAC_tnt.ATAF1_col_a]   

                         accession                     common title  
gene id   dna source                                                 
AT1G01060 col         [GSM1925527]        MYBrelated_tnt.LHY1_col_a  
          colamp      [GSM1925526]     MYBrelated_tnt.LHY1_colamp_a  
AT1G01250 col         [GSM1925016]     AP2EREBP_tnt.AT1G01250_col_a  
          colamp      [GSM1925015]  AP2EREBP_tnt.AT1G01250_colamp_a  
AT1G01720 col         [GSM1925578]              NAC_tnt.ATAF1_col_a

In [29]:
with open(parsed_data_dir / "dapseq_tfs.tsv", "w") as out:
    for x in set(df.index.get_level_values(0)):
        out.write(f"{x}\n")

# Download files...

In [30]:
if DOWNLOAD_NCBI_FILES:  
    
    from ftplib import FTP
    from datetime import datetime
    import tarfile

    ftp_link = series_data['esummary'][0]['FTPLink']; ftp_link
    
    url_parts = urllib.parse.urlparse(ftp_link)
    ftp_base = url_parts.netloc
    ftp_path = url_parts.path
    ftp_base, ftp_path
    
    ftp = FTP(ftp_base)
    ftp.login('','')
    ftp.cwd(f'/{ftp_path}/suppl/')  
    
    start = datetime.now()
    
    # Get All Files
    files = ftp.nlst()
    
    # Print out the files
    for file in files:
        if (raw_data_dir / SERIES_ACCESSION / file).exists():
            print(file, "already downloaded.")
        else:
            print("Downloading..." + file)
            ftp.retrbinary("RETR " + file, open( raw_data_dir / SERIES_ACCESSION / file, 'wb').write)
    
    end = datetime.now()
    diff = end - start
    
    ftp.quit()
    
    for f in (raw_data_dir / SERIES_ACCESSION).glob('*'):
        if f.suffix == ".tar":
            tar = tarfile.open(f)
            tar.extractall(raw_data_dir / SERIES_ACCESSION)
            tar.close()

    # Add downloaded file names to metadata
    def file_name(x):
        if len(x["accession"]) > 1:
            prefix = SERIES_ACCESSION
        else:
            prefix = x["accession"][0]
        
        file_name = raw_data_dir / SERIES_ACCESSION / f"{prefix}_DAPSeq-{x['common title'].replace('.', '-')}-chr1-5_GEM_events.narrowPeak.gz"
    
        if file_name.exists():
            return file_name
        else:
            print(file_name)
            return None
    
    df["file_name"] = df.apply(file_name, axis=1)


In [31]:
df['title'] = df['title'].apply(lambda x: ','.join(sorted(x)))
df['accession'] = df['accession'].apply(lambda x: ','.join(sorted(x)))

## Save metadata

In [32]:
df.head()

summary                 taxon    protein  \
gene id   dna source                                                
AT1G01060 col         young leaf  Arabidopsis thaliana       LHY1   
          colamp      young leaf  Arabidopsis thaliana       LHY1   
AT1G01250 col         young leaf  Arabidopsis thaliana  AT1G01250   
          colamp      young leaf  Arabidopsis thaliana  AT1G01250   
AT1G01720 col         young leaf  Arabidopsis thaliana      ATAF1   

                     protein family protein affinity tag expression system  \
gene id   dna source                                                         
AT1G01060 col            MYBrelated                 Halo               tnt   
          colamp         MYBrelated                 Halo               tnt   
AT1G01250 col              AP2EREBP                 Halo               tnt   
          colamp           AP2EREBP                 Halo               tnt   
AT1G01720 col                   NAC                 Halo               tnt   

                          subset replicate                            title  \
gene id   dna source                                                          
AT1G01060 col         motif+TFBS         a        MYBrelated_tnt.LHY1_col_a   
          colamp      motif+TFBS         a     MYBrelated_tnt.LHY1_colamp_a   
AT1G01250 col         motif+TFBS         a     AP2EREBP_tnt.AT1G01250_col_a   
          colamp           motif         a  AP2EREBP_tnt.AT1G01250_colamp_a   
AT1G01720 col         motif+TFBS         a              NAC_tnt.ATAF1_col_a   

                       accession                     common title  
gene id   dna source                                               
AT1G01060 col         GSM1925527        MYBrelated_tnt.LHY1_col_a  
          colamp      GSM1925526     MYBrelated_tnt.LHY1_colamp_a  
AT1G01250 col         GSM1925016     AP2EREBP_tnt.AT1G01250_col_a  
          colamp      GSM1925015  AP2EREBP_tnt.AT1G01250_colamp_a  
AT1G01720 col         GSM1925578              NAC_tnt.ATAF1_col_a

In [33]:
df.to_csv(parsed_data_dir / f"series_{SERIES_ACCESSION}_sample_metadata.tsv", sep="\t")

In [34]:
SERIES_ACCESSION

'GSE60141'

In [35]:
!head ../parsed_data/sample_metadata.tsv

gene id	dna source	summary	taxon	protein	protein family	protein affinity tag	expression system	replicate	title	accession	common title	file_name
AT1G01060	col	young leaf	Arabidopsis thaliana	LHY1	MYBrelated	Halo	tnt	a	MYBrelated_tnt.LHY1_col_a	GSM1925527	MYBrelated_tnt.LHY1_col_a	../raw_data/GSE60141/GSM1925527_DAPSeq-MYBrelated_tnt-LHY1_col_a-chr1-5_GEM_events.narrowPeak.gz
AT1G01060	colamp	young leaf	Arabidopsis thaliana	LHY1	MYBrelated	Halo	tnt	a	MYBrelated_tnt.LHY1_colamp_a	GSM1925526	MYBrelated_tnt.LHY1_colamp_a	../raw_data/GSE60141/GSM1925526_DAPSeq-MYBrelated_tnt-LHY1_colamp_a-chr1-5_GEM_events.narrowPeak.gz
AT1G01250	col	young leaf	Arabidopsis thaliana	AT1G01250	AP2EREBP	Halo	tnt	a	AP2EREBP_tnt.AT1G01250_col_a	GSM1925016	AP2EREBP_tnt.AT1G01250_col_a	../raw_data/GSE60141/GSM1925016_DAPSeq-AP2EREBP_tnt-AT1G01250_col_a-chr1-5_GEM_events.narrowPeak.gz
AT1G01250	colamp	young leaf	Arabidopsis thaliana	AT1G01250	AP2EREBP	Halo	tnt	a	AP2EREBP_tnt.AT1G01250_colamp_a	GSM1925015	AP2EREBP_tn

In [36]:
# END

### A check that the files match Ecker files... 

In [ ]:
path = "../../omalley2016_dapseq/not-used/dap_download_may2016_peaks.zip"
import os
import zipfile

ecker_file_titles = set()
for item in zipfile.ZipFile(path).namelist():
    ecker_file_titles.update([".".join(item.split("/")[2:4])])

In [ ]:
len(ecker_file_titles)

In [ ]:
df[df["common title"].isin(ecker_file_titles)].shape[0]

In [ ]:
df[~df["common title"].isin(ecker_file_titles)]